In [1]:
import torch

# Tensors
Tensors are the building block of PyTorch — every input, weight, activation, and gradient is a tensor.

In [2]:
# pattern 1.  Python list to Tensor
data = [
    [0, 0, 1, 1, 1, 1, 1, 1, 0, 0],
    [0, 1, 1, 0, 0, 0, 0, 1, 1, 0],
    [1, 1, 0, 1, 0, 0, 1, 0, 1, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    [1, 0, 1, 0, 0, 0, 1, 0, 0, 1],
    [1, 0, 0, 1, 1, 1, 1, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    [1, 1, 0, 0, 0, 0, 0, 0, 1, 1],
    [0, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 1, 1, 1, 1, 1, 1, 0, 0]
]
tensors = torch.tensor(data)
print(tensors)

tensor([[0, 0, 1, 1, 1, 1, 1, 1, 0, 0],
        [0, 1, 1, 0, 0, 0, 0, 1, 1, 0],
        [1, 1, 0, 1, 0, 0, 1, 0, 1, 1],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 1],
        [1, 0, 1, 0, 0, 0, 1, 0, 0, 1],
        [1, 0, 0, 1, 1, 1, 1, 0, 0, 1],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 1],
        [1, 1, 0, 0, 0, 0, 0, 0, 1, 1],
        [0, 1, 1, 1, 1, 1, 1, 1, 1, 0],
        [0, 0, 1, 1, 1, 1, 1, 1, 0, 0]])


In [3]:
# pattern 2: creating from a desired shape
# you specify the shape, not the values — this is how you initialize model weights
shape = (2, 3) # a shape tuple for 2 rows and three columns
ones = torch.ones(shape)
zeros = torch.zeros(shape)
random = torch.randn(shape)

print("Ones Tensor", ones)
print("Zeros Tensor", zeros)
print("Random Tensor", random)

Ones Tensor tensor([[1., 1., 1.],
        [1., 1., 1.]])
Zeros Tensor tensor([[0., 0., 0.],
        [0., 0., 0.]])
Random Tensor tensor([[ 0.7246,  1.4162, -0.7996],
        [ 0.6588, -0.6410, -0.0354]])


In [4]:
# pattern 3 creation by mimicking another tensor
# you need a new tensors with exact shape and size
template =  torch.tensor([[1, 2],[3, 4]])
rand_like = torch.randn_like(template, dtype=torch.float)
print("template Tensor", template)
print("rand_like Tensor", rand_like)

template Tensor tensor([[1, 2],
        [3, 4]])
rand_like Tensor tensor([[-0.3023,  0.6618],
        [ 0.0408, -0.4987]])


# What's inside a tensor
shape, dtype, device

In [5]:
tensor = torch.randn(2, 3)
print(tensor.shape) # 90% of errors are shape mismatch
print(tensor.dtype) # data type of numbers
print(tensor.device) # where does tensor live

torch.Size([2, 3])
torch.float32
cpu


`torch.float32` is the default dtype for floating-point tensors — a general precision/performance convention across deep learning frameworks, not something specific to gradients. It does matter *for* gradients though: training nudges weights by very small amounts each step, and `float32` has enough precision to represent those small updates without losing them — `float16` can, for example, underflow a small gradient to exactly zero. Autograd is what computes those gradients; the dtype is what determines how precisely they (and the resulting weight updates) can be represented.

# Autograd
- stands for automatic differentiation
- it's torch's built-in gradient calculator
- your model parameters (weights and biases) must be a float type — `float32` is standard.
  Data that represents categories or counts can stay integer.
- it requires being activated by setting `requires_grad = True`

- by default a tensor is just data; to tell PyTorch it's a learnable parameter you must set
  `requires_grad = True` — the single most important setting in all of PyTorch.
- setting it sends a message to the autograd engine: "this is a parameter — from now on,
  track every operation that happens to it."

# data vs parameters


In [6]:
import torch

# A standard data tensor
x_data = torch.tensor([[1., 2.],
                       [3., 4.]])

# A parameter tensor (we need gradients)
w = torch.tensor([[1.0],
                  [2.0]], requires_grad=True)

print(f"Data tensor requires_grad: {x_data.requires_grad}")
print(f"Parameter tensor requires_grad: {w.requires_grad}")

Data tensor requires_grad: False
Parameter tensor requires_grad: True


Once we set `requires_grad=True`, PyTorch starts a live recording of every operation done on that tensor.

# Building the graph
Goal: compute `z = x * y`, where `y = a + b`

In [7]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
x = torch.tensor(4.0, requires_grad=True)

y = a + b
z = x * y

print(z) # pytorch creates a graph for each operation

tensor(20., grad_fn=<MulBackward0>)


In [8]:
print(z.grad_fn) # print graph for z
print(y.grad_fn) # print graph for y
print(a.grad_fn) # None — a is a leaf tensor, not the result of an operation

None


In [9]:
def print_graph(fn, indent=0):
    if fn is None:
        return

    print(" " * indent + str(fn))

    for next_fn, _ in fn.next_functions:
        print_graph(next_fn, indent + 4)


print_graph(z.grad_fn)  # grad_fn is the breadcrumb back through the graph

## Tensor: noun
## Autograd: the nervous system

## Now we need operations — the verbs of torch (the actions/calculations)

## `*` vs `@`

In [10]:
import torch

# Element-wise multiplication
# Each element is multiplied with the corresponding element.

a = torch.tensor([
    [1, 2],
    [3, 4]
])

b = torch.tensor([
    [10, 20],
    [30, 40]
])

# Element-wise multiplication
result = a * b

print("a:")
print(a)

print("\nb:")
print(b)

print("\na * b:")
print(result)

a:
tensor([[1, 2],
        [3, 4]])

b:
tensor([[10, 20],
        [30, 40]])

a * b:
tensor([[ 10,  40],
        [ 90, 160]])


In [11]:
# 2. Matrix multiplication — @ powers the neural network linear algebra rule

import torch

m1 = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])

m2 = torch.tensor([
    [10, 20],
    [30, 40],
    [50, 60]
])

print("m1 shape:", m1.shape)
print("m2 shape:", m2.shape)

result = m1 @ m2

print("\nm1 @ m2:")
print(result)

print("\nResult shape:", result.shape)

m1 shape: torch.Size([2, 3])
m2 shape: torch.Size([3, 2])

m1 @ m2:
tensor([[220, 280],
        [490, 640]])

Result shape: torch.Size([2, 2])


## Building a linear layer with the classic formula `y = xW + b` uses `@`

## Reduction operations and the `dim` argument

A linear layer's raw output (`y = xW + b`) is one number **per example, per class** — e.g.
a `(2, 3)` tensor for a batch of 2 examples scored against 3 classes. That's not yet a
usable answer: a loss function needs one total number, and a prediction needs one class
per example, not 3 separate scores. **Reduction operations collapse a tensor along one
axis into fewer values** — `sum`/`mean` collapse into an aggregate, `max`/`argmax` collapse
into "the winner, and where it is."

`dim` says *which axis gets collapsed*. For a `(rows, cols)` tensor:
- `dim=0` walks **down** each column, collapsing the row axis — one result per column.
- `dim=1` walks **across** each row, collapsing the column axis — one result per row.

So `sum(dim=1)` on a `(2, 3)` tensor removes dim 1 (the column axis) and leaves one value
per row — a `(2,)` result, one total per example.

The four reductions used constantly in classification code:

| Op | Returns | Used for |
|---|---|---|
| `sum` | total of the values along `dim` | e.g. total loss across a batch |
| `mean` | average of the values along `dim` | e.g. average loss per batch |
| `max` | the **largest value** along `dim`, *and* its index — returned together as a `(values, indices)` pair | when you need both the winning score and which class it belongs to |
| `argmax` | **only the index** of the largest value along `dim` — not the value itself | when you only need the predicted class label, which is almost always what you actually want |

`argmax` matters specifically because a classifier's raw output is a score *per class*,
and "the prediction" means "whichever class has the highest score." `argmax(dim=1)` on a
`(batch, num_classes)` tensor returns exactly that: one integer class index per row — e.g.
`[1, 0]` below means "row 0's winning class is index 1, row 1's winning class is index 0."
That index is what gets compared against the true label to check if the model was right.

In [ ]:
scores = torch.tensor([
    [2.0, 5.0, 1.0],
    [7.0, 0.5, 3.0],
])

print("scores:\n", scores)
print("\nsum over everything:", scores.sum())
print("sum dim=0 (collapse rows, down each column):", scores.sum(dim=0))
print("sum dim=1 (collapse columns, across each row):", scores.sum(dim=1))
print("\nmean dim=1:", scores.mean(dim=1))

max_vals, max_idx = scores.max(dim=1)
print("\nmax dim=1 -> values:", max_vals, " indices:", max_idx)
print("argmax dim=1 (predicted class per row, in classification):", scores.argmax(dim=1))

# keepdim=True keeps the collapsed axis around as size 1 instead of dropping it entirely —
# matters when the result needs to still broadcast against the original (batch, classes)
# tensor, e.g. dividing each row by its own sum.
print("\nshape with keepdim=False:", scores.sum(dim=1, keepdim=False).shape)
print("shape with keepdim=True: ", scores.sum(dim=1, keepdim=True).shape)

## Softmax: turning raw scores into probabilities

`argmax` above gives a **hard decision** — just the index of the winning class, with no
sense of *how confident* that decision was. A row of raw scores like `[2.0, 5.0, 1.0]` and
one like `[2.0, 5.0, 4.9]` both `argmax` to class 1, but the first is a clear win and the
second is nearly a tie between classes 1 and 2 — `argmax` alone can't tell those two cases
apart. Raw scores themselves aren't usable as confidence either: they aren't bounded to
`[0, 1]` and don't sum to anything meaningful (`2.0 + 5.0 + 1.0 = 8.0` here, but a
different row could sum to anything), so they can't be compared or interpreted as
probabilities as-is.

**Softmax fixes this** by turning a row of raw scores into a genuine probability
distribution, in two steps:
1. **Exponentiate** every score (`e^score`) — this makes every value positive, and because
   `e^x` grows faster than `x`, it *disproportionately* boosts the largest score relative
   to the others (a small lead in raw score becomes a much bigger lead after exponentiating
   — this is the "with the largest score dominating the distribution" part).
2. **Normalize** by dividing each exponentiated score by the row's total — this is exactly
   the `sum(dim=1, keepdim=True)` reduction from above, forcing each row to sum to 1.

The output is now a real probability per class. Critically, `argmax` on these
probabilities always agrees with `argmax` on the raw scores — exponentiating (always
increasing) and dividing by a positive constant (the row sum) never changes *which* value
is largest, only the scale. So softmax doesn't change the prediction `argmax` already
gave you; it adds the magnitude/confidence information `argmax` was missing — the
"5.0 vs 4.9" near-tie above becomes visibly close in probability space (e.g. ~55% vs ~45%),
while a genuinely confident row stays confident (e.g. ~95% vs ~5%).

In [13]:
import torch.nn.functional as F

probs = F.softmax(scores, dim=1)   # dim=1 -> normalize across each row's classes
print("probs:\n", probs)
print("\neach row sums to 1:", probs.sum(dim=1))
print("argmax on probs matches argmax on raw scores:", probs.argmax(dim=1), scores.argmax(dim=1))

probs:
 tensor([[0.0466, 0.9362, 0.0171],
        [0.9806, 0.0015, 0.0180]])

each row sums to 1: tensor([1.0000, 1.0000])
argmax on probs matches argmax on raw scores: tensor([1, 0]) tensor([1, 0])


## `cat` vs `stack`: assembling a batch

Both combine tensors — the difference is whether a *new* dimension is created. `cat` joins
along an *existing* dimension (no new axis); `stack` creates a *new* dimension and lines
tensors up along it. This is the actual mechanism behind turning individual examples into a
batch: a `DataLoader` collating N separate `(3, 224, 224)` images into one `(N, 3, 224, 224)`
batch tensor is `torch.stack`, not `torch.cat`.

In [14]:
img1 = torch.zeros(3, 4, 4)   # pretend: one 3-channel, 4x4 image
img2 = torch.ones(3, 4, 4)    # another one

print("cat dim=0 shape:  ", torch.cat([img1, img2], dim=0).shape)     # (6, 4, 4) — no new axis, just longer
print("stack dim=0 shape:", torch.stack([img1, img2], dim=0).shape)   # (2, 3, 4, 4) — new "batch" axis

# this is exactly how a batch of N images gets built from a list of individual images:
batch = torch.stack([img1, img2])
print("\nbatch of 2 images:", batch.shape, "-> (batch_size, channels, height, width)")

cat dim=0 shape:   torch.Size([6, 4, 4])
stack dim=0 shape: torch.Size([2, 3, 4, 4])

batch of 2 images: torch.Size([2, 3, 4, 4]) -> (batch_size, channels, height, width)


## Indexing and slicing — same rules as NumPy

Used constantly in ML code: pulling a single example out of a batch, grabbing one channel
out of an image, slicing a window out of a sequence.

In [15]:
print("whole batch:", batch.shape)
print("first image only, batch[0]:", batch[0].shape)          # drop the batch dim
print("first channel of every image, batch[:, 0]:", batch[:, 0].shape)
print("a 2x2 crop of the first image, batch[0, :, :2, :2]:", batch[0, :, :2, :2].shape)

whole batch: torch.Size([2, 3, 4, 4])
first image only, batch[0]: torch.Size([3, 4, 4])
first channel of every image, batch[:, 0]: torch.Size([2, 4, 4])
a 2x2 crop of the first image, batch[0, :, :2, :2]: torch.Size([3, 2, 2])


# From Autograd to a Real Network

We now have the two ingredients autograd needs: tensors that carry `requires_grad=True`,
and operations that build a graph. But so far every example has been 2-3 loose tensors
(`a`, `b`, `x`) we tracked by hand. **A real network has thousands to billions of these —
manually naming and tracking each one doesn't scale.** That's the gap the next brick fills.

## Building a layer, by hand first

We already wrote `y = xW + b` using `@`. Let's actually run it with real, gradient-tracked
weights — then see what PyTorch gives us for free once we stop doing this by hand.

In [16]:
x = torch.tensor([[1.0, 2.0, 3.0]])
w = torch.randn(3, 4, requires_grad=True)
b = torch.randn(4, requires_grad=True)

y_manual = x @ w + b
print("manual y:", y_manual)

y_manual.sum().backward()
print("w.grad shape:", w.grad.shape, "b.grad shape:", b.grad.shape)
# this works, but imagine doing this for 50 layers — 100 separate w/b tensors to name,
# initialize, and pass to an optimizer by hand. that's the actual problem nn.Module solves.

manual y: tensor([[-0.5782,  0.7264, -0.1513, -3.0938]], grad_fn=<AddBackward0>)
w.grad shape: torch.Size([3, 4]) b.grad shape: torch.Size([4])


## `nn.Linear`: the same math, but PyTorch owns the bookkeeping

`nn.Linear(in_features, out_features)` creates and tracks the exact `w`/`b` pair we just
built by hand — same `y = xW + b`, same `requires_grad=True`, but now the weights live
*inside an object* PyTorch knows how to find, move to a device, and hand to an optimizer.

In [17]:
import torch.nn as nn

layer = nn.Linear(3, 4)

# prove it's doing the exact same math — copy our manual weights in and compare
with torch.no_grad():
    layer.weight.copy_(w.t())   # nn.Linear stores weight as (out_features, in_features) — transposed vs our manual w
    layer.bias.copy_(b)

y_module = layer(x)
print("nn.Linear y:", y_module)
print("matches manual y:", torch.allclose(y_manual, y_module))
print("\nlayer.weight.shape:", layer.weight.shape, "<- (out_features, in_features), not (in, out)")

nn.Linear y: tensor([[-0.5782,  0.7264, -0.1513, -3.0938]], grad_fn=<AddmmBackward0>)
matches manual y: True

layer.weight.shape: torch.Size([4, 3]) <- (out_features, in_features), not (in, out)


## Why stacking `nn.Linear` layers alone doesn't help

Natural next question: if one `nn.Linear` is a layer, does stacking several make a
"deeper," more powerful network? **No — not without something nonlinear between them.**
`y = W2(W1x + b1) + b2` algebraically collapses to `y = (W2W1)x + (W2b1 + b2)` — still just
one linear transform, `W_combined x + b_combined`. Provable, not just asserted:

In [18]:
layer1 = nn.Linear(3, 5)
layer2 = nn.Linear(5, 2)
data = torch.randn(4, 3)

stacked_out = layer2(layer1(data))

# collapse the two layers into one equivalent linear transform, algebraically
W_combined = layer2.weight @ layer1.weight
b_combined = layer2.weight @ layer1.bias + layer2.bias
collapsed_out = data @ W_combined.t() + b_combined

print("2 stacked linear layers == 1 collapsed linear layer:",
      torch.allclose(stacked_out, collapsed_out, atol=1e-6))

# now put a ReLU between them
out_with_relu = layer2(torch.relu(layer1(data)))
print("same trick, WITH relu in between:",
      torch.allclose(out_with_relu, collapsed_out, atol=1e-6), "<- breaks. ReLU is what makes depth matter.")

2 stacked linear layers == 1 collapsed linear layer: True
same trick, WITH relu in between: False <- breaks. ReLU is what makes depth matter.


## A multilayer network (MLP): `nn.Module` subclass

`nn.Module` is the base class for a whole network, not just one layer — subclass it,
register layers in `__init__` (assigning `self.fc1 = nn.Linear(...)` auto-registers it,
since `nn.Module` overrides `__setattr__` to recognize sub-layers), define the forward
pass in `forward()`. This is what "the network" actually means in code.

In [19]:
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 8)
        self.fc2 = nn.Linear(8, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))   # <- the nonlinearity from the cell above, now doing real work
        return self.fc2(x)

model = TinyNet()
print("parameters PyTorch found automatically, just from self.fc1/self.fc2 assignment:")
for name, p in model.named_parameters():
    print(" ", name, tuple(p.shape))

parameters PyTorch found automatically, just from self.fc1/self.fc2 assignment:
  fc1.weight (8, 3)
  fc1.bias (8,)
  fc2.weight (1, 8)
  fc2.bias (1,)


## Dataset + DataLoader: before we can train, we need batches

The model takes a batch of examples in one forward call (we already know why — `cat`
vs `stack` earlier is literally how a batch gets assembled). `Dataset` just needs to know
"how many examples" (`__len__`) and "give me example i" (`__getitem__`); `DataLoader` does
the batching, shuffling, and — for a real dataset — the parallel loading.

In [20]:
from torch.utils.data import Dataset, DataLoader

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# toy regression target: y = sum(x) + a little noise
X = torch.randn(50, 3)
y = X.sum(dim=1, keepdim=True) + torch.randn(50, 1) * 0.1

dataset = ToyDataset(X, y)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

for i, (xb, yb) in enumerate(loader):
    print(f"batch {i}: xb.shape={xb.shape} yb.shape={yb.shape}")
# 50 / 16 -> 3 full batches + 1 short batch of 2. the last batch being a different size
# is normal — worth knowing before it surprises you inside a training loop.

batch 0: xb.shape=torch.Size([16, 3]) yb.shape=torch.Size([16, 1])
batch 1: xb.shape=torch.Size([16, 3]) yb.shape=torch.Size([16, 1])
batch 2: xb.shape=torch.Size([16, 3]) yb.shape=torch.Size([16, 1])
batch 3: xb.shape=torch.Size([2, 3]) yb.shape=torch.Size([2, 1])


## Loss: the model makes predictions, but "wrong how much"?

Autograd can compute gradients of *anything* scalar — but "anything" needs to actually be
defined. Loss is that definition: a single number saying how wrong the current predictions
are. No loss, no `.backward()` target, no gradient, no learning.

## Optimizer: autograd computes the gradient, it doesn't apply it

`loss.backward()` fills in `.grad` on every parameter. Nothing has changed the weights yet
— that's the optimizer's one job: read `.grad`, apply an update rule (`AdamW` here — same
decoupled-weight-decay optimizer as everywhere else in this workspace), and only then do
the weights actually move.

## The training loop: every brick above, in one cycle

`zero_grad → forward → loss → backward → step`, repeated. This is the entire mechanism —
every bigger model in this workspace runs this exact five-line cycle, just with more
layers and a fancier loss.

In [21]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.05)

losses = []
for epoch in range(30):
    for xb, yb in loader:                # each epoch = one full pass over the DataLoader
        optimizer.zero_grad()            # clear grads from the last step
        pred = model(xb)                 # forward pass
        loss = loss_fn(pred, yb)         # how wrong right now
        loss.backward()                  # compute gradients
        optimizer.step()                 # actually update the weights
        losses.append(loss.item())

print("first batch loss:", losses[0])
print("last batch loss: ", losses[-1])

first batch loss: 2.205120086669922
last batch loss:  0.020950492471456528


## Device placement: why `.to(device)` at all

The loop above ran entirely on CPU — fine for 50 toy examples, not fine once a model has
millions of parameters and data has millions of rows. A GPU (or Apple Silicon's MPS) does
the same matmuls in parallel across thousands of cores instead of one at a time. The rule
that actually matters: **the model and the data it's fed must live on the same device** —
mixing a CPU model with a GPU tensor (or vice versa) is a real, common error, not a
performance nitpick.

In [22]:
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model.to(device)          # moves every parameter to the device, in place
X_dev = X.to(device)      # data has to move too — separately, it's not automatic

pred = model(X_dev)
print("prediction lives on:", pred.device)

device: mps


prediction lives on: mps:0


## Saving and loading: training is expensive, don't redo it

`state_dict()` is just the plain dict of every parameter tensor (we already met this on
`nn.Module` — same mechanism). Save that dict, not the model object itself; reload it into
a fresh model instance with the same architecture.

In [23]:
save_path = "/tmp/tiny_net_demo.pt"   # a real project would use its own checkpoints/ dir
torch.save(model.state_dict(), save_path)

fresh_model = TinyNet().to(device)          # a NEW, randomly-initialized instance
fresh_model.load_state_dict(torch.load(save_path))
fresh_model.eval()

with torch.no_grad():
    same = torch.allclose(model(X_dev), fresh_model(X_dev))
print("reloaded model gives identical predictions to the trained one:", same)

reloaded model gives identical predictions to the trained one: True


## Where to keep going

Every brick is now in place: Tensors (data) → Autograd (gradients) → Operations (math) →
`nn.Module` (organized parameters) → Loss + Optimizer (learning signal + weight updates) →
DataLoader + training loop (the repeatable cycle) → device + save/load (making it real).

The next natural question the video likely asks: what changes when the model is too big
for one GPU, or training needs to run across several? That's covered hands-on, with real
measured numbers on this same machine, in
[`mini-llms-playground/from_scratch/tinystories-gpt-6m/docs/EFFICIENT_TRAINING.md`](../../mini-llms-playground/from_scratch/tinystories-gpt-6m/docs/EFFICIENT_TRAINING.md)
(mixed precision, gradient checkpointing) and
[`.../docs/DISTRIBUTED_TRAINING.md`](../../mini-llms-playground/from_scratch/tinystories-gpt-6m/docs/DISTRIBUTED_TRAINING.md)
(DDP/FSDP) — the exact same `zero_grad → forward → loss → backward → step` loop from this
notebook, just with more machinery wrapped around it.